# SmolVLA × 日本語 × 2軸：GPU事前検証

**このノートブックは未実行のColab検証手順です。** 先に公開モデルの重み・前処理・2自由度の行動出力を検証し、実機購入の判断材料にします。実行にはGPU・インターネットが必要です。

**合格条件:** (1) 動画付き2軸LeRobotDatasetが作成・再読込できる、(2) SmolVLAをロードできる、(3) カメラ画像と日本語指示を入力し、最後の次元が2の行動チャンクを生成できる。**それだけでは日本語理解や実機での指示成功は証明されません。**

**重要:** 事前学習済みモデルは元のロボット用入出力特徴を保持しているため、2軸の features を明示的に設定します。公式 issue: https://github.com/huggingface/lerobot/issues/2418

In [ ]:
import sys, subprocess, torch
print("Python:", sys.version)
print("Torch:", torch.__version__, "GPU:", torch.cuda.is_available())
assert torch.cuda.is_available(), "GPUランタイムを選択してください"
print(torch.cuda.get_device_name(0))
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "lerobot[smolvla,training]==0.6.1", "pillow"])
print("Ready")

## 1. 実機の代わりに極小の2軸教師データを作成

**重要なデータ設計:** 同じ画像に対して「赤」と「青」の二つの指示と異なる正解軌道を記録します。言語を無視して画像だけから角度を暗記できないようにします。画像は人工的な丸、角度は教材用の幾何教師が計算する値です。

In [ ]:
import numpy as np
from PIL import Image, ImageDraw
from pathlib import Path
from lerobot.datasets.lerobot_dataset import LeRobotDataset

DATA_ROOT = Path("/content/pointer-vla-mini")
REPO_ID = "local/pointer_vla_mini"
features = {
    "observation.state": {"dtype": "float32", "shape": (2,), "names": ["pan", "tilt"]},
    "action": {"dtype": "float32", "shape": (2,), "names": ["pan", "tilt"]},
    "observation.images.front": {"dtype": "video", "shape": (96, 96, 3), "names": ["height","width","channels"]},
}
dataset = LeRobotDataset.create(repo_id=REPO_ID, root=DATA_ROOT, fps=10,
                                features=features, robot_type="pico_pan_tilt",
                                use_videos=True, image_writer_threads=2)
rng = np.random.default_rng(20261017)
for scene in range(6):
    red = (int(rng.integers(18,45)), int(rng.integers(18,76)))
    blue = (int(rng.integers(53,80)), int(rng.integers(18,76)))
    image = Image.new("RGB",(96,96),(235,236,230))
    draw=ImageDraw.Draw(image)
    for (x,y),color in [(red,"#eb4040"),(blue,"#3560ed")]:
        draw.ellipse((x-7,y-7,x+7,y+7), fill=color)
    image_np=np.asarray(image, dtype=np.uint8)
    for color,xy,task in [("red",red,"赤い丸を指して"),("blue",blue,"青い丸を指して")]:
        target=np.asarray([90+(xy[0]/95-0.5)*85, 90-(xy[1]/95-0.5)*65],dtype=np.float32)
        start=np.asarray([90.,90.], dtype=np.float32)
        prev=start.copy()
        for step in range(12):
            u=(step+1)/12
            s=u*u*(3-2*u)
            action=(start+(target-start)*s).astype(np.float32)
            dataset.add_frame({"observation.state": prev.copy(),
                               "action": action.copy(),
                               "observation.images.front": image_np.copy(),
                               "task": task})
            prev=action
        dataset.save_episode()
dataset.finalize()
dataset = LeRobotDataset(repo_id=REPO_ID, root=DATA_ROOT)
print("dataset frames",len(dataset),"features",dataset.meta.features.keys())
assert dataset.meta.features["observation.state"]["shape"] == [2] or tuple(dataset.meta.features["observation.state"]["shape"]) == (2,)
assert len(dataset) >= 100
print("PASS: mini LeRobotDataset generated and reloaded")

## 2. 事前学習済みSmolVLAで「2軸出力」の本物の推論を実行

ここではモデルを再学習しません。学習済み重みをダウンロードして、2軸用の入出力特徴と今回のデータセット統計に合わせ、行動チャンクを生成します。成功しても角度が正しいとは限りません。

In [ ]:
import torch
from lerobot.configs import FeatureType, PolicyFeature
from lerobot.policies.smolvla import SmolVLAPolicy
from lerobot.policies.factory import make_pre_post_processors

MODEL_ID = "lerobot/smolvla_base"
device=torch.device("cuda")
policy=SmolVLAPolicy.from_pretrained(MODEL_ID).to(device).eval()
policy.config.input_features = {
    "observation.state": PolicyFeature(type=FeatureType.STATE,shape=(2,)),
    "observation.images.front": PolicyFeature(type=FeatureType.VISUAL,shape=(3,96,96)),
}
policy.config.output_features = {"action":PolicyFeature(type=FeatureType.ACTION,shape=(2,))}
assert policy.config.max_state_dim >= 2
assert policy.config.max_action_dim >= 2

preprocess, postprocess=make_pre_post_processors(
    policy.config, dataset_stats=dataset.meta.stats)
sample=dataset[0]
batch={
    "observation.state": sample["observation.state"].unsqueeze(0),
    "observation.images.front": sample["observation.images.front"].unsqueeze(0),
    "task":["赤い丸を指して"],
}
torch.manual_seed(20261017)
with torch.inference_mode():
    result=policy.predict_action_chunk(preprocess(batch))
print("SmolVLA raw normalized chunk shape:",tuple(result.shape))
assert result.ndim == 3 and result.shape[0]==1 and result.shape[-1]==2
assert torch.isfinite(result).all()
print("PASS: actual SmolVLA accepted image + Japanese task and generated 2-axis continuous action chunk")
print("CAVEAT: this does NOT prove Japanese instruction grounding or successful pointing.")

## 3. 次にすべきこと

前セルがPASSしたら、実機データ収集・モデル追加学習を進められます。モデルの性能評価は、未学習の画像配置で赤／青の指示を切り替え、教師に依存せず実機で目標を指すか調べる必要があります。ノートブックの出力ログとGPU種類を記録してください。